In [ ]:
import json
import torch
from torch import nn
from pathlib import Path
from torch_geometric.nn import GATConv
from torch.nn import Linear, ModuleList, ReLU
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data, InMemoryDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TemporalDependencyDataset(InMemoryDataset):
    def __init__(self, path, tokenizer, model, max_length=64):
        super().__init__()
        self.tokenizer = tokenizer
        self.encoder = model.to(device)
        self.max_length = max_length
        self.data_list = self.load_graphs(path)
        self.data, self.slices = self.collate(self.data_list)

    def encode_text(self, text):
        tokens = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            output = self.encoder(**tokens).last_hidden_state
        return output[:, 0, :].squeeze(0).cpu()

    def encode_event(self, event_text):
        tokens = self.tokenizer(event_text, return_tensors="pt", truncation=True, max_length=self.max_length).to(device)
        with torch.no_grad():
            output = self.encoder(**tokens).last_hidden_state
        return output[:, 0, :].squeeze(0).cpu()

    def load_graphs(self, path):
        data_list = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                entry = json.loads(line)
                context_emb = self.encode_text(entry["text"])
                node_feats = []
                id_to_idx = {}
                for idx, event in enumerate(entry["events"]):
                    emb = self.encode_event(event["text"])
                    node_feats.append(emb)
                    id_to_idx[event["id"]] = idx

                x = torch.stack(node_feats)
                edge_index = []
                edge_label = []
                context = context_emb.repeat(len(entry["events"])**2, 1)
                for target in entry["events"]:
                    deps = target["depends_on"]
                    if deps is None:
                        continue
                    if isinstance(deps, str):
                        deps = [deps]
                    for src_id in deps:
                        edge_index.append([id_to_idx[src_id], id_to_idx[target["id"]]])
                        edge_label.append(1)

                num_nodes = len(entry["events"])
                all_edges = set((i, j) for i in range(num_nodes) for j in range(num_nodes) if i != j)
                pos_edges = set(tuple(e) for e in edge_index)
                neg_edges = list(all_edges - pos_edges)
                for i, j in neg_edges[:len(edge_index)]:
                    edge_index.append([i, j])
                    edge_label.append(0)

                edge_index = torch.tensor(edge_index).t().contiguous()
                edge_label = torch.tensor(edge_label, dtype=torch.float)
                data_list.append(Data(x=x, edge_index=edge_index, edge_label=edge_label, context=context_emb))
        return data_list

class TemporalGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.convs = ModuleList([
    		GATConv(in_channels, hidden_channels, heads=1, concat=True),
    		GATConv(hidden_channels, hidden_channels, heads=1, concat=True)
        ])
        self.relu = ReLU()
        self.classifier = nn.Sequential(
			Linear(hidden_channels * 2 + in_channels, 128),
			ReLU(),
			Linear(128, 1))

    def forward(self, x, edge_index, edge_pairs, context):
        for conv in self.convs:
            x = self.relu(conv(x, edge_index))
        row, col = edge_pairs
        edge_feats = torch.cat([x[row], x[col], context.expand(row.size(0), -1)], dim=1)
        return self.classifier(edge_feats).squeeze(1)

model_name = "bert-base-cased"
encoder = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder.eval()

path = Path("dataset.jsonl")
dataset = TemporalDependencyDataset(path, tokenizer, encoder)

train_data = dataset[:160]
val_data = dataset[160:]
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
val_loader = DataLoader(val_data, batch_size=1)

In [ ]:
model = TemporalGNN(in_channels=768, hidden_channels=256).to(device)
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(encoder.parameters()),
    lr=1e-4
)
criterion = torch.nn.BCEWithLogitsLoss()

def compute_metrics(y_true, y_pred):
    y_true = torch.tensor(y_true)
    y_pred = torch.tensor(y_pred)
    tp = ((y_true == 1) & (y_pred == 1)).sum().item()
    fp = ((y_true == 0) & (y_pred == 1)).sum().item()
    fn = ((y_true == 1) & (y_pred == 0)).sum().item()
    tn = ((y_true == 0) & (y_pred == 0)).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    return precision, recall, f1, accuracy

In [ ]:
best_f1 = 0
for epoch in range(2_000):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        edge_pairs = batch.edge_index
        context = batch.context.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, edge_pairs, context)
        loss = criterion(out, batch.edge_label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            edge_pairs = batch.edge_index
            context = batch.context.to(device)
            out = model(batch.x, batch.edge_index, edge_pairs, context)
            pred = torch.sigmoid(out) > 0.5
            y_true.extend(batch.edge_label.cpu().tolist())
            y_pred.extend(pred.cpu().tolist())

    p, r, f1, acc = compute_metrics(y_true, y_pred)
    print(f"Validation — Precision: {p:.3f}, Recall: {r:.3f}, F1: {f1:.3f}, Accuracy: {acc:.3f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pt")
        print("model_saved!")
